# Temporal consistency and calibration

Executable assertions check source timing, chronological partitions, train-only transformations and finite-sample calibration.

In [1]:
from pathlib import Path
import sys
_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dataset').is_dir() and (p / 'notebooks').is_dir())
if str(_root / 'notebooks') not in sys.path:
    sys.path.insert(0, str(_root / 'notebooks'))
from _runtime import (reuse as _nb_reuse, source_document as _source_document,
                      initialize, repo_root as _repo_root, result_tables,
                      result_state, plot_data, figure_stem)
initialize()

from pathlib import Path
import json, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
PROJECT_DIR=_repo_root()
TABLE_DIR=PROJECT_DIR/'.cache/experiment/tables'; STATE_DIR=PROJECT_DIR/'.cache/experiment/state'; FIGURE_DIR=PROJECT_DIR/'.cache/experiment/figures'
pd.set_option('display.max_rows',100);pd.set_option('display.max_columns',20);pd.set_option('display.max_colwidth',None)
plt.rcParams.update({'font.family':'Times New Roman','font.size':18,'axes.titlesize':20,'axes.labelsize':18,'xtick.labelsize':18,'ytick.labelsize':18,'legend.fontsize':18,'figure.dpi':350,'savefig.dpi':350})
manifest=pd.read_csv(TABLE_DIR/'raw_meter_manifest.csv')

protocol=json.loads((STATE_DIR/'experiment_config.json').read_text())
base_panel=pd.read_parquet(TABLE_DIR/'causal_base_panel.parquet')
modeling=pd.read_parquet(TABLE_DIR/'modeling_dataset.parquet')
split_table=pd.read_csv(TABLE_DIR/'split_summary.csv',parse_dates=['start','end'])


## Canonical fold-local transformation under test
The transformation is defined once in Notebook 02 and imported below. Dynamic availability and future-perturbation tests exercise that same function.

In [2]:
_nb_reuse('src-02-012', globals())


## Chronological partitions and future-perturbation invariance

In [3]:
def check(name,condition,detail):
    assert bool(condition),f'{name}: {detail}'
check('four disjoint chronological splits',list(split_table.split)==['train','validation','calibration','test'] and all(split_table.end.iloc[i]<split_table.start.iloc[i+1] for i in range(3)),'Global common-calendar boundaries')
for h in protocol['horizons']:
    valid=modeling[f'target_h{h}_net_load_kw'].notna()
    check(f'target boundary h={h}',(modeling.loc[valid,f'target_timestamp_h{h}']<=modeling.loc[valid,'split_end']).all(),'Targets crossing all split boundaries are purged')
    check(f'target horizon h={h}',(modeling[f'target_timestamp_h{h}']-modeling.timestamp).eq(pd.Timedelta(minutes=15*h)).all(),'Exact horizon in calendar evaluation time')
check('raw source availability',(modeling.max_source_timestamp.dropna()<=modeling.loc[modeling.max_source_timestamp.notna(),'timestamp']).all(),'Every input source snapshot timestamp is at or before forecast origin')
check('neighbor context availability',(modeling.graph_context_timestamp<=modeling.timestamp).all(),'No neighbor timestamp t+h enters forecast inputs')
state=json.loads((STATE_DIR/'main_preprocessing_state.json').read_text())
check('targets excluded from feature dictionary',not any(c.startswith('target_') or 'supplier' in c or c=='raw_observed_net_load_kw' for c in state['feature_columns']),'Only explicit eligible features enter models')
# Perturb future raw values after validation start, retaining the fitted training history.
cut=split_table.loc[split_table.split.eq('validation'),'start'].iloc[0]+pd.Timedelta(days=2)
perturbed=base_panel.copy();future=perturbed.timestamp>cut
for col in ['net_load_kw','pv_power_kw','storage_charge_kw','storage_decharge_kw','ev_kw','heat_pump_kw','supplier_target_reference_kw']:
    if col in perturbed:perturbed.loc[future,col]=999999.
original,original_state=build_fold(base_panel,split_table)
changed,changed_state=build_fold(perturbed,split_table)
columns=state['feature_columns']
pd.testing.assert_frame_equal(original.loc[original.timestamp<=cut,columns].reset_index(drop=True),changed.loc[changed.timestamp<=cut,columns].reset_index(drop=True))
check('future perturbation invariance',True,'Changing every later input/secondary target cannot change any earlier feature after preprocessing is fitted')
for key in ['cap','auxiliary_caps','graph_weights','train_start','train_end']:
    check('train-only '+key,original_state[key]==changed_state[key],'No validation/calibration/test value affects fitted preprocessing state')
lagged,_=build_fold(base_panel,split_table,graph_delay=1)
check('lagged graph delay',(lagged.graph_context_timestamp==lagged.timestamp-pd.Timedelta(minutes=15)).all(),'One-step-lagged context is explicit')
check('nonnegative quality weights',modeling.imputed_value_confidence_weight.between(.03,1).all(),'Weights depend on contemporaneous causal missingness')


## Conformal score and exact finite-sample order statistic
The nonnegative two-sided score is max(lower − y, y − upper, 0). Correction uses the ceil((n+1)(1−alpha))-th ordered score; an unavailable order statistic returns infinity. Exchangeability is not assumed to hold automatically for these time series: block and rolling calibration diagnostics are still required.

In [4]:
_nb_reuse('src-03-005', globals(), names=['conformity_score'])

_nb_reuse('src-03-005', globals(), names=['finite_sample_correction'])

np.testing.assert_array_equal(conformity_score([5,1,9],[2,2,2],[8,8,8]),[0,1,1])
np.testing.assert_array_equal(conformity_score([2,8],[2,2],[8,8]),[0,0])
assert finite_sample_correction(np.arange(10),.1)==9
assert finite_sample_correction(np.arange(19),.1)==17
assert np.isinf(finite_sample_correction([0,1],.1))
for invalid in [([], .1),([1],0),([np.nan],.1)]:
    try:finite_sample_correction(*invalid)
    except ValueError:pass
    else:raise AssertionError('Invalid sample accepted')
check('conformal score unit tests',True,'Inside, boundary, below and above interval')
check('finite-sample quantile unit tests',True,'Exact order statistic, small-n infinity, invalid inputs')
(STATE_DIR/'temporal_consistency.json').write_text(json.dumps({'status':'PASS','protocol_revision':protocol['revision']},indent=2))


## Rapid-ramp semantics: both directions and subthreshold behavior

In [5]:
ramp_data=pd.read_parquet(TABLE_DIR/'modeling_dataset.parquet',columns=['site','net_load_ramp_kw','ramp_scale','rapid_ramp_flag'])
negative=ramp_data.net_load_ramp_kw.lt(0)&ramp_data.net_load_ramp_kw.abs().gt(ramp_data.ramp_scale)
positive=ramp_data.net_load_ramp_kw.gt(ramp_data.ramp_scale)&ramp_data.net_load_ramp_kw.gt(0)
quiet=ramp_data.net_load_ramp_kw.abs().lt(ramp_data.ramp_scale)
assert negative.any() and positive.any() and quiet.any()
assert ramp_data.loc[negative,'rapid_ramp_flag'].eq(1).all(),'Large negative ramps must be detected'
assert ramp_data.loc[positive,'rapid_ramp_flag'].eq(1).all(),'Large positive ramps must be detected'
assert ramp_data.loc[quiet,'rapid_ramp_flag'].eq(0).all(),'Subthreshold ramps must not be flagged'
counts=pd.DataFrame([{'case':'negative_above_absolute_threshold','n':int(negative.sum())},{'case':'positive_above_threshold','n':int(positive.sum())},{'case':'below_absolute_threshold','n':int(quiet.sum())}])
(STATE_DIR/'ramp_definition.json').write_text(json.dumps({'status':'PASS','rule':'abs(net_load[t]-net_load[t-1]) >= train-only site 85th percentile of absolute differences','tie_convention':'inclusive','units':'kW difference per grid step; not divided by current net load','tested_signs':['positive','negative'],'threshold_source':'train only'},indent=2))
display(counts)


,case,n
0,negative_above_absolute_threshold,15885
1,positive_above_threshold,15947
2,below_absolute_threshold,167233
